# Per-Subdomain Highlight Sentiment Classifier

Does prediction accuracy vary by **subdomain**, and can subdomain-specific modeling
improve on the universal baseline?

**Dataset**: 16 subdomains, 1–57 selections each. Sparse subdomains (n<5) flagged throughout.

**Three analyses**:
1. Universal model (RF, 5-fold CV) — predictions disaggregated by subdomain post-hoc
2. Leave-One-Subdomain-Out (LOSO) CV — cross-subdomain generalization
3. Subdomain-conditioned few-shot Claude (batch) — examples from same subdomain; sparse fallback to same domain

**Batch workflow** (phases 1–4) used for analysis 3.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy as scipy_entropy

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error
import anthropic
from dotenv import dotenv_values

DATA_DIR       = Path('../data-exports/20260412_183830/highlight_analysis_output')
MOD_FILE       = Path('../data-exports/20260412_183830/'
                      'moderation_sessions_export_20260412_183830.csv')
OUT_DIR        = DATA_DIR / 'classifier_output'
OUT_DIR.mkdir(exist_ok=True)

CACHE_FILE     = OUT_DIR / 'llm_predictions_cache.json'
BATCH_IDS_FILE = OUT_DIR / 'batch_ids.json'

RANDOM_STATE   = 42
OUTER_K        = 5
SPARSE_THRESH  = 5
CLAUDE_MODEL   = 'claude-opus-4-7'
POLL_INTERVAL  = 30

try:
    BEST_K = int(
        pd.read_csv(OUT_DIR / 'cv_fewshot_binary.csv')
        .pipe(lambda d: d.loc[d['AUC (conf)'].idxmax(), 'k'])
    )
except FileNotFoundError:
    BEST_K = 4
    print("cv_fewshot_binary.csv not found — using BEST_K=4.")

print(f"BEST_K: {BEST_K}  |  Model: {CLAUDE_MODEL}")

BEST_K: 8  |  Model: claude-opus-4-7


## Load data

In [2]:
df_sel = pd.read_csv(DATA_DIR / 'df_sel.csv')
mod    = pd.read_csv(MOD_FILE)
scenario_text = mod.drop_duplicates('scenario_id')[['scenario_id', 'scenario_prompt', 'original_response']]
df = df_sel.merge(scenario_text, on='scenario_id', how='left').reset_index(drop=True)

subdomains = df['subdomain'].value_counts().sort_values()
print(subdomains.to_string())
sparse_sds = subdomains[subdomains < SPARSE_THRESH].index.tolist()
print(f"\nSparse (n<{SPARSE_THRESH}): {sparse_sds}")

subdomain
Human Nature             1
Politics                 4
Family                   6
STEM                     7
Internet Interaction     9
Academic Standing       10
Community Engagement    10
Protective Measures     11
Behavioral Norms        12
Friendship              12
Finance                 13
Self                    26
HW                      28
Health                  30
Privacy                 31
Common Sense            57

Sparse (n<5): ['Human Nature', 'Politics']


In [3]:
y_binary     = (df['highlight_sentiment'] >= 5).astype(int).values
y_rating_raw = df['highlight_sentiment'].astype(int).values


def build_structured_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df_in.index)
    out['strategy_is_null'] = df_in['model_strategy'].isna().astype(int)
    out = pd.concat([out,
        pd.get_dummies(df_in['model_strategy'].fillna('none'), prefix='strat'),
        pd.get_dummies(df_in['parent_motivation'].fillna('unknown'), prefix='motiv'),
    ], axis=1)
    for col in ['domain', 'age_band', 'sensitivity_level', 'relationship_frame',
                'space_type', 'trait', 'trait_level']:
        out = pd.concat([out, pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)], axis=1)
    out['breakdown_expected'] = (df_in['breakdown_expected'] == 'yes').astype(int)
    for col in ['parent_gender', 'parent_age_group', 'parent_education', 'parent_ethnicity',
                'area_of_residency', 'child_has_ai_use', 'parent_llm_monitoring_level']:
        out = pd.concat([out, pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)], axis=1)
    out['genai_regular_user'] = (df_in['genai_familiarity'] == 'regular_user').astype(int)
    freq_map = {'daily': 2, 'weekly': 1, 'monthly_or_less': 0}
    out['genai_usage_freq'] = df_in['genai_usage_frequency'].map(freq_map).fillna(0).astype(int)
    out['parent_internet_use_frequency'] = pd.to_numeric(
        df_in['parent_internet_use_frequency'], errors='coerce').fillna(0)
    out['is_only_child'] = (df_in['is_only_child'].astype(str).str.lower() == 'yes').astype(int)
    for s in 'ABCD':
        out[f'parenting_{s}'] = df_in['parenting_style'].fillna('').str.contains(s).astype(int)
    return out.astype(float)


X_struct = build_structured_features(df)
print(f"Structured features: {X_struct.shape[1]}")

Structured features: 82


## Part 1 — Universal model, per-subdomain breakdown

Standard 5-fold CV; out-of-fold predictions disaggregated by subdomain.

In [4]:
try:
    cv = StratifiedKFold(n_splits=OUTER_K, shuffle=True, random_state=RANDOM_STATE)
    list(cv.split(np.zeros(len(df)), y_binary))
except ValueError:
    cv = KFold(n_splits=OUTER_K, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.ones(len(df), dtype=int)
oof_proba = np.full(len(df), 0.685)
rf        = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE)

for train_idx, test_idx in cv.split(np.zeros(len(df)), y_binary):
    rf.fit(X_struct.values[train_idx], y_binary[train_idx])
    oof_preds[test_idx] = rf.predict(X_struct.values[test_idx])
    oof_proba[test_idx] = rf.predict_proba(X_struct.values[test_idx])[:, 1]

universal_rows = []
for sd in df['subdomain'].unique():
    mask   = (df['subdomain'] == sd).values
    y_true = y_binary[mask]
    try:
        auc = roc_auc_score(y_true, oof_proba[mask]) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        auc = np.nan
    universal_rows.append({
        'subdomain':     sd,
        'n':             mask.sum(),
        'sparse':        mask.sum() < SPARSE_THRESH,
        'pct_pos':       round(y_true.mean(), 3),
        'majority_f1':   round(f1_score(y_true, np.ones_like(y_true), zero_division=0), 3),
        'F1_universal':  round(f1_score(y_true, oof_preds[mask], zero_division=0), 3),
        'AUC_universal': round(auc, 3) if not np.isnan(auc) else np.nan,
    })

universal_df = pd.DataFrame(universal_rows).sort_values('n')
print(universal_df.to_string(index=False))
universal_df.to_csv(OUT_DIR / 'per_subdomain_universal.csv', index=False)

           subdomain  n  sparse  pct_pos  majority_f1  F1_universal  AUC_universal
        Human Nature  1    True    0.000        0.000         0.000            NaN
            Politics  4    True    0.000        0.000         0.000            NaN
              Family  6   False    0.667        0.800         0.889          0.500
                STEM  7   False    0.714        0.833         0.727          0.600
Internet Interaction  9   False    0.444        0.615         0.667          0.900
   Academic Standing 10   False    0.900        0.947         0.947          1.000
Community Engagement 10   False    0.900        0.947         0.947          1.000
 Protective Measures 11   False    0.545        0.706         0.462          0.233
    Behavioral Norms 12   False    0.917        0.957         0.957          1.000
          Friendship 12   False    0.250        0.400         0.857          1.000
             Finance 13   False    0.692        0.818         0.875          0.972
    

## Part 2 — Leave-One-Subdomain-Out (LOSO) CV

In [5]:
loso_rows = []
for sd in df['subdomain'].unique():
    test_mask  = (df['subdomain'] == sd).values
    y_te       = y_binary[test_mask]
    n          = test_mask.sum()

    if n < 2 or len(np.unique(y_te)) < 2:
        loso_rows.append({'subdomain': sd, 'n': n, 'F1_loso': np.nan, 'AUC_loso': np.nan,
                          'note': 'insufficient test data'})
        continue

    rf_loso = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE)
    rf_loso.fit(X_struct.values[~test_mask], y_binary[~test_mask])
    y_pred  = rf_loso.predict(X_struct.values[test_mask])
    y_proba = rf_loso.predict_proba(X_struct.values[test_mask])[:, 1]
    try:
        auc = roc_auc_score(y_te, y_proba)
    except Exception:
        auc = np.nan
    loso_rows.append({
        'subdomain': sd, 'n': n,
        'F1_loso':   round(f1_score(y_te, y_pred, zero_division=0), 3),
        'AUC_loso':  round(auc, 3) if not np.isnan(auc) else np.nan,
        'note':      'sparse' if n < SPARSE_THRESH else '',
    })

loso_df = pd.DataFrame(loso_rows).sort_values('n')
print(loso_df.to_string(index=False))
loso_df.to_csv(OUT_DIR / 'per_subdomain_loso.csv', index=False)

           subdomain  n  F1_loso  AUC_loso                   note
        Human Nature  1      NaN       NaN insufficient test data
            Politics  4      NaN       NaN insufficient test data
              Family  6    0.889     0.500                       
                STEM  7    0.889     0.800                       
Internet Interaction  9    0.571     0.650                       
   Academic Standing 10    0.947     1.000                       
Community Engagement 10    1.000     1.000                       
 Protective Measures 11    0.545     0.367                       
    Behavioral Norms 12    1.000     1.000                       
          Friendship 12    0.429     0.222                       
             Finance 13    0.778     0.806                       
                Self 26    0.788     0.635                       
                  HW 28    0.913     0.952                       
              Health 30    0.894     0.665                       
          

## Part 2b — LOSO + per-subdomain history features

Uses the **test subdomain's own other selections** as history features (within-subdomain LOO),
mirroring the per-participant approach in `2_per_parent.ipynb`.

For each test row of subdomain S, features are computed from S's *other* selections
(excluding the row being predicted). Training rows use all other same-subdomain rows
in the training set.

4 new features (80 → 84 total):
- `sd_mean_sentiment` — mean of other selections in same subdomain
- `sd_pct_positive` — fraction rated ≥5
- `sd_strategy_entropy` — Shannon entropy of model strategies highlighted
- `sd_n_prior` — count of other selections in same subdomain

Fallback when a subdomain has only 1 row: use domain-level stats from the training fold.

In [6]:
class MeanRegressor:
    def fit(self, X, y):  self.mean_ = y.mean(); return self
    def predict(self, X): return np.full(len(X), self.mean_)

loso_clf_models = {
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=8,
                                             random_state=RANDOM_STATE),
}
loso_reg_models = {
    'Mean Baseline': MeanRegressor(),
    'Ridge':         Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=8,
                                            random_state=RANDOM_STATE),
}


def _sd_features_for_rows(df_rows: pd.DataFrame, fallback: dict) -> np.ndarray:
    """
    For each row in df_rows, compute history features from all OTHER rows with the same
    subdomain (within df_rows), excluding the current row by index identity.
    Fallback is used when a row has no same-subdomain neighbors.
    """
    result = []
    for i, (idx, row) in enumerate(df_rows.iterrows()):
        sd     = row['subdomain']
        others = df_rows[(df_rows['subdomain'] == sd) & (df_rows.index != idx)]
        if len(others) == 0:
            result.append([
                fallback['mean_sent'],
                fallback['pct_pos'],
                fallback['strat_ent'],
                0,
            ])
        else:
            sc  = others['model_strategy'].value_counts(normalize=True)
            ent = scipy_entropy(sc.values) if len(sc) > 1 else 0.0
            result.append([
                others['highlight_sentiment'].mean(),
                (others['highlight_sentiment'] >= 5).mean(),
                ent,
                len(others),
            ])
    return np.array(result)


def _make_fallback(df_pool: pd.DataFrame, subdomain: str) -> dict:
    """Domain-level fallback for sparse subdomains; global fallback if domain also missing."""
    domain = df_pool[df_pool['subdomain'] == subdomain]['domain'].iloc[0] \
        if (df_pool['subdomain'] == subdomain).any() else None
    pool = df_pool[df_pool['domain'] == domain] if domain else df_pool
    if len(pool) == 0:
        pool = df_pool
    sc = pool['model_strategy'].value_counts(normalize=True)
    return {
        'mean_sent': pool['highlight_sentiment'].mean(),
        'pct_pos':   (pool['highlight_sentiment'] >= 5).mean(),
        'strat_ent': scipy_entropy(sc.values) if len(sc) > 1 else 0.0,
    }


def run_loso_with_per_subdomain(df, X_struct, y_binary, models: dict) -> list[dict]:
    rows = []
    for sd in df['subdomain'].unique():
        test_mask = (df['subdomain'] == sd).values
        df_tr     = df[~test_mask].reset_index(drop=True)
        df_te     = df[test_mask].reset_index(drop=True)
        y_te      = y_binary[test_mask]

        if len(y_te) < 2 or len(np.unique(y_te)) < 2:
            for name in models:
                rows.append({'subdomain': sd, 'n': len(y_te), 'model': name,
                             'F1_sd': np.nan, 'AUC_sd': np.nan, 'note': 'insufficient'})
            continue

        fb   = _make_fallback(df_tr, sd)
        sd_tr = _sd_features_for_rows(df_tr, fb)
        sd_te = _sd_features_for_rows(df_te, fb)
        X_tr  = np.hstack([X_struct[~test_mask], sd_tr])
        X_te  = np.hstack([X_struct[test_mask],  sd_te])

        for name, model in models.items():
            model.fit(X_tr, y_binary[~test_mask])
            y_pred  = model.predict(X_te)
            y_proba = model.predict_proba(X_te)[:, 1] \
                if hasattr(model, 'predict_proba') else y_pred.astype(float)
            try:
                auc = roc_auc_score(y_te, y_proba)
            except Exception:
                auc = np.nan
            rows.append({
                'subdomain': sd, 'n': len(y_te), 'model': name,
                'F1_sd':  round(f1_score(y_te, y_pred, zero_division=0), 3),
                'AUC_sd': round(auc, 3) if not np.isnan(auc) else np.nan,
                'note':   'sparse' if len(y_te) < SPARSE_THRESH else '',
            })
    return pd.DataFrame(rows)


def run_loso_with_per_subdomain_rating(df, X_struct, y_rating, models: dict) -> list[dict]:
    rows = []
    for sd in df['subdomain'].unique():
        test_mask = (df['subdomain'] == sd).values
        df_tr     = df[~test_mask].reset_index(drop=True)
        df_te     = df[test_mask].reset_index(drop=True)
        y_te      = y_rating[test_mask]

        fb    = _make_fallback(df_tr, sd)
        sd_tr = _sd_features_for_rows(df_tr, fb)
        sd_te = _sd_features_for_rows(df_te, fb)
        X_tr  = np.hstack([X_struct[~test_mask], sd_tr])
        X_te  = np.hstack([X_struct[test_mask],  sd_te])

        for name, model in models.items():
            model.fit(X_tr, y_rating[~test_mask])
            y_pred = np.clip(np.round(model.predict(X_te)).astype(int), 1, 7)
            mae    = mean_absolute_error(y_te, y_pred)
            r      = np.corrcoef(y_te, y_pred)[0, 1] if len(y_te) > 1 else np.nan
            rows.append({
                'subdomain':  sd, 'n': len(y_te), 'model': name,
                'MAE_sd':     round(mae, 3),
                'Pearson_r_sd': round(r, 3) if not np.isnan(r) else np.nan,
                'note':       'sparse' if len(y_te) < SPARSE_THRESH else '',
            })
    return pd.DataFrame(rows)


# --- Base LOSO rating (no extra features) for comparison ---
loso_base_rating_rows = []
for sd in df['subdomain'].unique():
    test_mask = (df['subdomain'] == sd).values
    y_te      = y_rating_raw[test_mask]
    for name, model in loso_reg_models.items():
        model.fit(X_struct.values[~test_mask], y_rating_raw[~test_mask])
        y_pred = np.clip(np.round(model.predict(X_struct.values[test_mask])).astype(int), 1, 7)
        mae    = mean_absolute_error(y_te, y_pred)
        r      = np.corrcoef(y_te, y_pred)[0, 1] if len(y_te) > 1 else np.nan
        loso_base_rating_rows.append({
            'subdomain': sd, 'n': len(y_te), 'model': name,
            'MAE_base': round(mae, 3),
            'r_base':   round(r, 3) if not np.isnan(r) else np.nan,
        })
loso_base_rating_df = pd.DataFrame(loso_base_rating_rows)
print("Base LOSO — rating 1–7 (mean across subdomains):")
print(loso_base_rating_df.groupby('model')[['MAE_base', 'r_base']].mean().round(3).to_string())
loso_base_rating_df.to_csv(OUT_DIR / 'per_subdomain_loso_base_rating.csv', index=False)

# --- LOSO + per-subdomain features ---
loso_sd = run_loso_with_per_subdomain(df, X_struct.values, y_binary, loso_clf_models)
print("\nLOSO + per-subdomain history — binary (mean across subdomains):")
print(loso_sd.groupby('model')[['F1_sd', 'AUC_sd']].mean().round(3).to_string())
print("\nComparison — LOSO RF binary (base vs. +per-subdomain features):")
base_rf = loso_df[['subdomain', 'F1_loso', 'AUC_loso']].set_index('subdomain')
sd_rf   = loso_sd[loso_sd['model'] == 'Random Forest'][['subdomain', 'F1_sd', 'AUC_sd']].set_index('subdomain')
cmp_bin = base_rf.join(sd_rf).round(3)
cmp_bin['ΔAUC'] = (cmp_bin['AUC_sd'] - cmp_bin['AUC_loso']).round(3)
print(cmp_bin.sort_values('ΔAUC').to_string())
loso_sd.to_csv(OUT_DIR / 'per_subdomain_loso_per_subdomain.csv', index=False)

loso_sd_rating = run_loso_with_per_subdomain_rating(df, X_struct.values, y_rating_raw, loso_reg_models)
print("\nLOSO + per-subdomain history — rating 1–7 (mean across subdomains):")
print(loso_sd_rating.groupby('model')[['MAE_sd', 'Pearson_r_sd']].mean().round(3).to_string())
loso_sd_rating.to_csv(OUT_DIR / 'per_subdomain_loso_per_subdomain_rating.csv', index=False)

# 3-way rating comparison: base LOSO | LOSO + per-sd features
base_r = loso_base_rating_df.groupby('model')[['MAE_base', 'r_base']].mean()
sd_r   = loso_sd_rating.groupby('model')[['MAE_sd', 'Pearson_r_sd']].mean()
cmp3   = base_r.join(sd_r)
cmp3.columns = ['MAE (LOSO base)', 'r (LOSO base)', 'MAE (+sd hist)', 'r (+sd hist)']
cmp3['ΔMAE'] = (cmp3['MAE (+sd hist)'] - cmp3['MAE (LOSO base)']).round(3)
cmp3['Δr']   = (cmp3['r (+sd hist)']   - cmp3['r (LOSO base)']).round(3)
cmp3         = cmp3.round(3)
print("\nRating task — LOSO base vs. +per-subdomain history (ΔMAE>0 = worse):")
print(cmp3.to_string())
cmp3.to_csv(OUT_DIR / 'per_subdomain_rating_comparison.csv')

Base LOSO — rating 1–7 (mean across subdomains):
               MAE_base  r_base
model                          
Mean Baseline     1.616     NaN
Random Forest     1.292   0.507
Ridge             1.710   0.327

LOSO + per-subdomain history — binary (mean across subdomains):
               F1_sd  AUC_sd
model                       
Random Forest  0.797   0.726

Comparison — LOSO RF binary (base vs. +per-subdomain features):
                      F1_loso  AUC_loso  F1_sd  AUC_sd   ΔAUC
subdomain                                                    
Internet Interaction    0.571     0.650  0.364   0.150 -0.500
Protective Measures     0.545     0.367  0.462   0.300 -0.067
Common Sense            0.814     0.704  0.742   0.685 -0.019
Family                  0.889     0.500  0.889   0.500  0.000
Academic Standing       0.947     1.000  0.941   1.000  0.000
Community Engagement    1.000     1.000  0.947   1.000  0.000
Behavioral Norms        1.000     1.000  0.957   1.000  0.000
Friendship      

## Part 3 — Subdomain-conditioned few-shot Claude (batch)

In [ ]:
env      = dotenv_values(Path('.') / '.env')
api_key  = env.get('CLAUDE_API_KEY')
base_url = env.get('CLAUDE_API_BASE_URL')

SKIP_CLAUDE = not api_key
if not SKIP_CLAUDE:
    client = anthropic.Anthropic(api_key=api_key, base_url=base_url)
    print(f"Client ready. Model: {CLAUDE_MODEL}")
else:
    print("WARNING: CLAUDE_API_KEY not found — skipping Claude cells.")


def load_cache() -> dict:
    return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

def save_cache(c: dict) -> None:
    CACHE_FILE.write_text(json.dumps(c, indent=2))

def load_batch_ids() -> dict:
    return json.loads(BATCH_IDS_FILE.read_text()) if BATCH_IDS_FILE.exists() else {}

def save_batch_ids(b: dict) -> None:
    BATCH_IDS_FILE.write_text(json.dumps(b, indent=2))

In [ ]:
SYSTEM_BINARY_SD = """You are an expert evaluator of AI responses to children's questions.
A parent reviewed an AI chatbot's response to their child's message and highlighted
a specific passage. They then rated that passage on a 1–7 sentiment scale where
1 = very negative or inappropriate and 7 = very positive or appropriate.

Your task is to predict whether the parent rated the highlighted passage as
POSITIVE (rating ≥5) or NEGATIVE/NEUTRAL (rating < 5).

You will be given labeled examples from the same topic subdomain, and the case to predict.

Respond ONLY with a JSON object in this exact format, no other text:
{"prediction": 0 or 1, "confidence": 0.0 to 1.0}"""

SYSTEM_RATING_SD = """You are an expert evaluator of AI responses to children's questions.
A parent reviewed an AI chatbot's response to their child's message and highlighted
a specific passage. They then rated that passage on a 1–7 sentiment scale where
1 = very negative or inappropriate and 7 = very positive or appropriate.

Your task is to predict the parent's 1–7 sentiment rating.
You will be given labeled examples from the same topic subdomain, and the case to predict.

Respond ONLY with a JSON object in this exact format, no other text:
{"rating": <integer 1-7>}"""


def make_case_block(row: pd.Series) -> str:
    return (f"AGE BAND: {row.get('age_band', 'unknown')}\n"
            f"DOMAIN: {row.get('domain', 'unknown')} / {row.get('subdomain', 'unknown')}\n\n"
            f"CHILD'S QUESTION:\n{row['scenario_prompt']}\n\n"
            f"FULL AI RESPONSE:\n{row['original_response']}\n\n"
            f"HIGHLIGHTED TEXT:\n{row['highlight_text']}")


def make_subdomain_prefix(
    df_train: pd.DataFrame, subdomain: str, domain: str,
    k: int, random_state: int = 42,
) -> str:
    pool          = df_train[df_train['subdomain'] == subdomain]
    fallback_note = f"subdomain '{subdomain}'"
    if len(pool) < k:
        pool          = df_train[df_train['domain'] == domain]
        fallback_note = f"domain '{domain}' (subdomain pool too small)"
    if len(pool) == 0:
        pool          = df_train
        fallback_note = 'full training set'

    neg, pos   = pool[pool['highlight_sentiment'] < 5], pool[pool['highlight_sentiment'] >= 5]
    k_neg, k_pos = k // 2, k - k // 2
    examples   = pd.concat([
        neg.sample(n=k_neg, replace=len(neg) < k_neg, random_state=random_state),
        pos.sample(n=k_pos, replace=len(pos) < k_pos, random_state=random_state),
    ]).sample(frac=1, random_state=random_state)

    parts = [f"SUBDOMAIN CONTEXT: {subdomain}\nEXAMPLES (from {fallback_note}):\n"]
    for i, (_, row) in enumerate(examples.iterrows(), 1):
        rating = int(row['highlight_sentiment'])
        label  = 'POSITIVE (≥5)' if rating >= 5 else 'NEGATIVE/NEUTRAL (<5)'
        parts.append(f"[EXAMPLE {i} — TRUE RATING: {rating} ({label})]\n" + make_case_block(row))
    parts.append("\nNow predict for the following:")
    return "\n\n".join(parts)

### Phase 1 — Prepare

In [ ]:
def prepare_subdomain_requests(df: pd.DataFrame, k: int, cache: dict) -> list[dict]:
    requests   = []
    subdomains = df['subdomain'].unique()

    for sd_idx, sd in enumerate(subdomains):
        test_mask = (df['subdomain'] == sd).values
        df_train  = df[~test_mask]
        df_test   = df[test_mask]
        test_pos  = np.where(test_mask)[0]
        domain    = df_test['domain'].iloc[0] if len(df_test) > 0 else ''
        prefix    = make_subdomain_prefix(df_train, sd, domain, k,
                                           random_state=sd_idx * 100 + k)

        for pos, (_, row) in zip(test_pos, df_test.iterrows()):
            user_msg = prefix + "\n\n" + make_case_block(row)

            for task, system in [('binary', SYSTEM_BINARY_SD),
                                  ('rating', SYSTEM_RATING_SD)]:
                key = f"subdomain_sd{sd_idx}_k{k}_{task}_pos{pos}"
                if key not in cache:
                    requests.append({
                        "custom_id": key,
                        "params": {
                            "model": CLAUDE_MODEL, "max_tokens": 64,
                            "system": [{"type": "text", "text": system,
                                        "cache_control": {"type": "ephemeral"}}],
                            "messages": [{"role": "user", "content": user_msg}],
                        },
                    })
    return requests


if not SKIP_CLAUDE:
    cache    = load_cache()
    requests = prepare_subdomain_requests(df, BEST_K, cache)
    print(f"Requests to submit: {len(requests)}  "
          f"(cached: {len(df) * 2 - len(requests)})")

### Phase 2 — Submit

In [ ]:
if not SKIP_CLAUDE and requests:
    batch      = client.messages.batches.create(requests=requests)
    batch_id   = batch.id
    batch_ids  = load_batch_ids()
    batch_ids['per_subdomain'] = batch_id
    save_batch_ids(batch_ids)
    print(f"Batch submitted: {batch_id}  ({len(requests)} requests)")
elif not SKIP_CLAUDE:
    print("All requests cached — skipping submission.")
    batch_id = load_batch_ids().get('per_subdomain')

### Phase 3 — Poll and save

In [ ]:
if not SKIP_CLAUDE:
    if 'batch_id' not in dir() or batch_id is None:
        batch_id = load_batch_ids().get('per_subdomain')
        if not batch_id:
            raise RuntimeError("No batch_id found — run the submit cell first.")
        print(f"Recovered batch_id: {batch_id}")

    print(f"Polling {batch_id} every {POLL_INTERVAL}s…")
    while True:
        batch  = client.messages.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"  {batch.processing_status} — "
              f"processing={counts.processing}  succeeded={counts.succeeded}  "
              f"errored={counts.errored}")
        if batch.processing_status == 'ended':
            break
        time.sleep(POLL_INTERVAL)

    cache    = load_cache()
    n_ok, n_err = 0, 0
    for result in client.messages.batches.results(batch_id):
        if result.result.type == 'succeeded':
            try:
                cache[result.custom_id] = json.loads(
                    result.result.message.content[0].text.strip()
                )
                n_ok += 1
            except Exception:
                cache[result.custom_id] = None
                n_err += 1
        else:
            cache[result.custom_id] = None
            n_err += 1

    save_cache(cache)
    print(f"Cache updated: {n_ok} succeeded, {n_err} failed.")

### Phase 4 — Extract predictions and evaluate

In [ ]:
if not SKIP_CLAUDE:
    cache        = load_cache()
    sd_list      = df['subdomain'].unique()
    binary_preds = np.ones(len(df), dtype=int)
    binary_conf  = np.full(len(df), 0.5)
    rating_preds = np.full(len(df), 5, dtype=int)

    for sd_idx, sd in enumerate(sd_list):
        for pos in np.where((df['subdomain'] == sd).values)[0]:
            b = cache.get(f"subdomain_sd{sd_idx}_k{BEST_K}_binary_pos{pos}")
            if b and 'prediction' in b:
                binary_preds[pos] = int(b['prediction'])
                binary_conf[pos]  = float(b.get('confidence', 0.5))
            r = cache.get(f"subdomain_sd{sd_idx}_k{BEST_K}_rating_pos{pos}")
            if r and 'rating' in r:
                rating_preds[pos] = int(np.clip(r['rating'], 1, 7))

    # Per-subdomain few-shot metrics
    fewshot_rows = []
    for sd in sd_list:
        mask   = (df['subdomain'] == sd).values
        y_true = y_binary[mask]
        bp, bc = binary_preds[mask], binary_conf[mask]
        try:
            auc = roc_auc_score(y_true, bc) if len(np.unique(y_true)) > 1 else np.nan
        except Exception:
            auc = np.nan
        fewshot_rows.append({
            'subdomain':   sd,
            'n':           mask.sum(),
            'F1_fewshot':  round(f1_score(y_true, bp, zero_division=0), 3),
            'AUC_fewshot': round(auc, 3) if not np.isnan(auc) else np.nan,
        })

    fewshot_df = pd.DataFrame(fewshot_rows)
    fewshot_df.to_csv(OUT_DIR / 'per_subdomain_fewshot.csv', index=False)

    # Consolidated summary
    summary = (
        universal_df[['subdomain', 'n', 'sparse', 'pct_pos', 'majority_f1',
                       'F1_universal', 'AUC_universal']]
        .merge(loso_df[['subdomain', 'F1_loso', 'AUC_loso', 'note']], on='subdomain', how='left')
        .merge(fewshot_df[['subdomain', 'F1_fewshot', 'AUC_fewshot']], on='subdomain', how='left')
        .sort_values('n')
    )
    auc_cols = ['AUC_universal', 'AUC_loso', 'AUC_fewshot']
    summary['best_approach'] = summary[auc_cols].idxmax(axis=1).str.replace('AUC_', '')

    print(summary.to_string(index=False))
    summary.to_csv(OUT_DIR / 'per_subdomain_summary.csv', index=False)

## Visualizations

In [ ]:
if not SKIP_CLAUDE:
    heatmap_data = (
        summary.set_index('subdomain')[['AUC_universal', 'AUC_loso', 'AUC_fewshot']]
        .rename(columns={'AUC_universal': 'Universal (5-fold)',
                          'AUC_loso':      'LOSO',
                          'AUC_fewshot':   f'Few-shot k={BEST_K}'})
        .sort_values('LOSO', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(9, max(6, len(heatmap_data) * 0.45)))
    sns.heatmap(heatmap_data.astype(float), annot=True, fmt='.3f',
                cmap='RdYlGn', vmin=0.4, vmax=1.0,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'AUC'})
    ax.set_title('Per-subdomain AUC by approach (sorted by LOSO AUC)')
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'per_subdomain_heatmap.png', dpi=100)
    plt.show()

    valid = summary.dropna(subset=['AUC_loso']).sort_values('n')
    x, w  = np.arange(len(valid)), 0.28
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w,   valid['AUC_universal'], w, label='Universal (5-fold)', color='steelblue')
    ax.bar(x,       valid['AUC_loso'],      w, label='LOSO',               color='darkorange')
    ax.bar(x + w,   valid['AUC_fewshot'],   w, label=f'Few-shot k={BEST_K}', color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{sd}\n(n={n})" for sd, n in zip(valid['subdomain'], valid['n'])],
        rotation=45, ha='right', fontsize=8,
    )
    ax.set_ylabel('AUC')
    ax.set_title('Per-subdomain AUC: Universal vs. LOSO vs. Few-shot (sorted by n)')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'per_subdomain_bar.png', dpi=100)
    plt.show()

    print("\n3 hardest subdomains by LOSO AUC:")
    print(valid.nsmallest(3, 'AUC_loso')[
        ['subdomain', 'n', 'AUC_universal', 'AUC_loso', 'AUC_fewshot']
    ].to_string(index=False))